# 01 Gold Label Audit

This notebook audits the isolated Gold label table. The label table is allowed
to contain post-race outcomes, but those outcomes must remain separate from
feature generation and only join at training dataset construction time.

In [1]:
from pathlib import Path
import json
from datetime import datetime

import numpy as np
import pandas as pd
import plotly.express as px

ROOT = Path.cwd()
while not (ROOT / "configs" / "pipeline_config.yaml").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent

NOTEBOOK_NAME = "01_gold_label_audit"
GOLD = ROOT / "data" / "gold"
FEATURES = GOLD / "features"
LABELS = GOLD / "labels"
TRAINING = GOLD / "training"
METADATA = GOLD / "metadata"

OUTPUT_TABLES = ROOT / "eda" / "gold" / "outputs" / "tables" / NOTEBOOK_NAME
OUTPUT_CHARTS = ROOT / "eda" / "gold" / "outputs" / "charts" / NOTEBOOK_NAME
OUTPUT_REPORTS = ROOT / "eda" / "gold" / "outputs" / "reports" / NOTEBOOK_NAME
INSIGHTS = ROOT / "eda" / "gold" / "insights"
CHECKPOINTS = ROOT / "eda" / "gold" / "checkpoints"
for path in [OUTPUT_TABLES, OUTPUT_CHARTS, OUTPUT_REPORTS, INSIGHTS, CHECKPOINTS]:
    path.mkdir(parents=True, exist_ok=True)

def read_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8"))

def write_report(name: str, payload: dict) -> None:
    (OUTPUT_REPORTS / f"{name}.json").write_text(json.dumps(payload, indent=2, default=str), encoding="utf-8")

def write_insight(title: str, observations: list[str], issues: list[str], recommendations: list[str]) -> None:
    content = f"# {title}\n\n"
    content += f"**Generated at:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n"
    content += "## Key Observations\n\n" + "\n".join(f"- {item}" for item in observations) + "\n\n"
    content += "## Issues\n\n" + ("\n".join(f"- {item}" for item in issues) if issues else "- None") + "\n\n"
    content += "## Recommendations\n\n" + "\n".join(f"- {item}" for item in recommendations) + "\n"
    (INSIGHTS / f"{NOTEBOOK_NAME}.md").write_text(content, encoding="utf-8")

def save_chart(fig, name: str) -> None:
    fig.write_html(OUTPUT_CHARTS / f"{name}.html", include_plotlyjs="cdn")

print("=" * 72)
print(f"GOLD EDA - {NOTEBOOK_NAME}")
print(f"Start time: {datetime.now()}")
print(f"Gold root: {GOLD}")


GOLD EDA - 01_gold_label_audit
Start time: 2026-06-02 14:28:21.473384
Gold root: D:\F1_WinRate_Predictor\data\gold


## Finish Bucket Distribution

F1 finish outcomes are naturally imbalanced: wins are rare, podiums are limited
to three cars, and points depend on sporting rules. The key audit question is
whether the label table preserves every driver-session outcome without leaking
those outcomes into feature artifacts.

In [2]:
labels = pd.read_parquet(LABELS / "race_result_labels.parquet")
label_contract = read_json(METADATA / "label_contract.json")

bucket_summary = (
    labels.groupby(["target_finish_bucket", "target_finish_bucket_label"], dropna=False)
    .size()
    .rename("rows")
    .reset_index()
    .sort_values("target_finish_bucket")
)
bucket_summary["pct"] = (bucket_summary["rows"] / bucket_summary["rows"].sum() * 100).round(2)
bucket_summary.to_csv(OUTPUT_TABLES / "finish_bucket_distribution.csv", index=False)

fig = px.bar(
    bucket_summary,
    x="target_finish_bucket_label",
    y="rows",
    color="target_finish_bucket_label",
    text="rows",
    title="Gold Label Distribution: Finish Bucket Target",
    labels={"target_finish_bucket_label": "Finish bucket", "rows": "Driver-session labels"},
)
fig.update_layout(showlegend=False, margin=dict(l=10, r=10, t=55, b=80))
fig.update_traces(textposition="outside", cliponaxis=False)
save_chart(fig, "finish_bucket_distribution")
fig.show()

display(bucket_summary)

,target_finish_bucket,target_finish_bucket_label,rows,pct
0,0,WIN,68,4.95
1,1,PODIUM_NON_WIN,136,9.90
2,2,POINTS_NON_PODIUM,446,32.46
3,3,CLASSIFIED_OUTSIDE_POINTS,562,40.90
4,4,DNF_DNS_DSQ_UNCLASSIFIED,162,11.79


## Label Integrity

The label grain must be exactly one row per `session_key + driver_number`. Any
duplicate key would create ambiguous supervision for the model, while nullable
targets would create silent training exclusions.

In [3]:
quality_rows = []
quality_rows.append({
    "check": "label_key_uniqueness",
    "metric": "duplicate session-driver rows",
    "value": int(labels.duplicated(["session_key", "driver_number"]).sum()),
    "status": "PASS" if labels.duplicated(["session_key", "driver_number"]).sum() == 0 else "FAIL",
})
for column in ["target_win", "target_podium", "target_top10", "target_points", "target_dnf", "target_finish_bucket"]:
    null_count = int(labels[column].isna().sum()) if column in labels.columns else len(labels)
    quality_rows.append({
        "check": f"{column}_not_null",
        "metric": "null target rows",
        "value": null_count,
        "status": "PASS" if null_count == 0 else "FAIL",
    })
label_quality = pd.DataFrame(quality_rows)
label_quality.to_csv(OUTPUT_TABLES / "label_quality_matrix.csv", index=False)

fig = px.bar(
    label_quality,
    x="check",
    y="value",
    color="status",
    title="Gold Label Quality Matrix",
    labels={"check": "Quality check", "value": "Issue count"},
)
fig.update_layout(margin=dict(l=10, r=10, t=55, b=120))
save_chart(fig, "label_quality_matrix")
fig.show()

display(label_quality)

,check,metric,value,status
0,label_key_uniqueness,duplicate session-driver rows,0,PASS
1,target_win_not_null,null target rows,0,PASS
2,target_podium_not_null,null target rows,0,PASS
3,target_top10_not_null,null target rows,0,PASS
4,target_points_not_null,null target rows,0,PASS
5,target_dnf_not_null,null target rows,0,PASS
6,target_finish_bucket_not_null,null target rows,0,PASS


In [4]:
write_report("label_audit", {
    "rows": int(len(labels)),
    "unique_sessions": int(labels["session_key"].nunique()),
    "unique_driver_sessions": int(labels[["session_key", "driver_number"]].drop_duplicates().shape[0]),
    "quality_status": "PASS" if label_quality["status"].eq("PASS").all() else "FAIL",
    "bucket_distribution": bucket_summary.to_dict(orient="records"),
})
write_insight(
    "Gold Label Audit",
    [
        f"Gold labels contain {len(labels):,} driver-session outcomes across {labels['session_key'].nunique():,} sessions.",
        "Target classes are intentionally imbalanced because F1 sporting outcomes are constrained by grid size and points rules.",
        "Post-race outcome fields remain isolated in the label artifact and are not present in master_lap_features.",
    ],
    [] if label_quality["status"].eq("PASS").all() else ["One or more label integrity checks failed."],
    [
        "Use stratified validation where possible because finish buckets are imbalanced.",
        "Keep label columns out of feature selection; join them only in training dataset generation.",
    ],
)
(CHECKPOINTS / "gold_label_audit_completed.txt").write_text(datetime.now().isoformat(), encoding="utf-8")

26